<a href="https://colab.research.google.com/github/Dilukshika-Sasitharan/Statistical-Learning-e23355/blob/main/E23355_Assignment_7c.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **ASSIGNMENT 7C : ITEM RESPONSE PREDICTION AND CLICK THROUGH RATE PREDICTION**


# **E/23/355**
---


# Q1. Bayesian Estimation of a User Ability Parameter from Item Responses

## Setup

An online learning platform presents a sequence of multiple-choice questions to a user.

Each response is modeled as

$$
Y_i \in \{0,1\}
$$

where

- **\(Y_i = 1\)** : Correct response
- **\(Y_i = 0\)** : Incorrect response

The probability of answering item \(i\) correctly follows the Two-Parameter Logistic (2PL) Item Response Theory model:

$$
P(Y_i=1 \mid \Theta=\theta)
=
p_i(\theta)
=
\frac{1}{1+\exp\left[-a_i(\theta-b_i)\right]}
$$

where

- \(a_i>0\) : Discrimination parameter
- \(b_i\) : Difficulty parameter
- \(\theta\) : Latent user ability

The prior distribution is

$$
\Theta \sim N(0,1)
$$

After every response, the posterior distribution obtained at the current step becomes the prior distribution for the next Bayesian update.

# Task 1 — Visualizing the Mechanics

The probability of answering an item correctly is visualized for the following parameter settings:

- \(a=0.5,\; b=0\)
- \(a=1.5,\; b=-2\)
- \(a=1.5,\; b=0\)
- \(a=1.5,\; b=2\)

## Interpretation

The difficulty parameter \(b_i\) shifts the logistic curve horizontally without changing its shape.

The curve always satisfies

$$
p_i(b_i)=0.5,
$$

which means that the probability of answering correctly is **50%** when the user's ability equals the item's difficulty.

- Increasing \(b_i\) shifts the curve to the **right**, indicating that greater ability is required to answer correctly.
- Decreasing \(b_i\) shifts the curve to the **left**, indicating that even users with lower ability have a reasonable probability of success.

The discrimination parameter \(a_i\) controls the steepness of the curve.

- Larger values of \(a_i\) produce a steeper S-shaped curve, allowing the item to distinguish more effectively between users with similar ability levels.
- Smaller values of \(a_i\) produce a flatter curve, making the item less sensitive to differences in ability.

In [1]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

# Generate theta values
theta_grid = np.linspace(0,1,500)

# Beta distributions
beta_configs = [

    {"alpha":1,"beta":1,
     "name":"Beta(1,1)",
     "color":"gray",
     "dash":"dash"},

    {"alpha":2,"beta":8,
     "name":"Beta(2,8)",
     "color":"orange",
     "dash":"solid"},

    {"alpha":8,"beta":2,
     "name":"Beta(8,2)",
     "color":"green",
     "dash":"solid"}

]

fig = go.Figure()

for config in beta_configs:

    pdf = stats.beta.pdf(
        theta_grid,
        config["alpha"],
        config["beta"]
    )

    fig.add_trace(

        go.Scatter(

            x=theta_grid,
            y=pdf,

            mode="lines",

            name=config["name"],

            line=dict(
                color=config["color"],
                dash=config["dash"],
                width=2.5
            )

        )

    )

fig.update_layout(

    title="Beta Distribution Probability Density Functions",

    xaxis_title="CTR Parameter (θ)",

    yaxis_title="Probability Density",

    template="plotly_white",

    hovermode="x unified"

)

fig.show()

# Task 2 — Sequential Likelihood Contribution

For a single response \(y_k\), the likelihood contribution is

$$
L(y_k \mid \theta)
=
p_k(\theta)^{y_k}
\left[1-p_k(\theta)\right]^{1-y_k}.
$$

Assuming that responses are conditionally independent given the latent ability \(\Theta=\theta\), the likelihood of the complete response history

$$
y^{(k)}=(y_1,y_2,\ldots,y_k)
$$

is

$$
L(y^{(k)} \mid \theta)
=
\prod_{i=1}^{k}
p_i(\theta)^{y_i}
\left[1-p_i(\theta)\right]^{1-y_i}.
$$

This likelihood summarizes all information contained in the user's responses up to step \(k\).

# Task 3 — Mathematical Formulation of the Running Update

Using Bayes' theorem, the posterior distribution after observing response \(y_k\) is

$$
f(\theta \mid y^{(k)})
=
\frac{
L(y_k \mid \theta)
\,f(\theta \mid y^{(k-1)})
}{
\int
L(y_k \mid s)
\,f(s \mid y^{(k-1)})
\,ds
}.
$$

Ignoring the normalizing constant, the recursive Bayesian update becomes

$$
f(\theta \mid y^{(k)})
\propto
L(y_k \mid \theta)
\,f(\theta \mid y^{(k-1)}).
$$

Substituting the Bernoulli likelihood gives

$$
f(\theta \mid y^{(k)})
\propto
p_k(\theta)^{y_k}
\left[1-p_k(\theta)\right]^{1-y_k}
\,f(\theta \mid y^{(k-1)}).
$$

Thus, the posterior distribution is obtained by multiplying the previous posterior distribution by the likelihood of the newest response.

After normalization, this posterior becomes the prior distribution for the next item.

# Task 4 — Dynamic Shifting

Suppose the user answers item \(k\) correctly (\(y_k=1\)) and the item has a large
difficulty parameter \(b_k\).

The likelihood contribution is

$$
L(y_k \mid \theta)=p_k(\theta),
$$

where

$$
p_k(\theta)=
\frac{1}
{1+\exp\left[-a_k(\theta-b_k)\right]}.
$$

Since \(b_k\) is large, the probability of a correct response is low for users with
small ability values and becomes high only when the latent ability \(\theta\) approaches
or exceeds \(b_k\).

Using Bayes' theorem,

$$
f(\theta \mid y^{(k)})
\propto
p_k(\theta)
\,f(\theta \mid y^{(k-1)}),
$$

the previous posterior distribution is multiplied by the likelihood of the newest response.

Consequently,

- Probability mass corresponding to low ability values decreases.
- Probability mass corresponding to high ability values increases.
- Both the posterior mean and the MAP estimate shift toward larger values of \(\theta\).

Therefore, correctly answering a highly difficult item provides strong evidence that the
user has high latent ability.

Conversely,

- An incorrect response to a difficult item causes only a small posterior change because
  such an outcome is already expected.
- An incorrect response to an easy item provides much stronger evidence of lower ability
  and shifts the posterior distribution toward smaller values of \(\theta\).

# Task 5 — Tracking Certainty and Sharpness

The discrimination parameter \(a_k\) determines how informative an item is during the
Bayesian updating process.

Since the posterior distribution is obtained by multiplying the previous posterior by the
likelihood,

$$
f(\theta \mid y^{(k)})
\propto
L(y_k \mid \theta)
\,f(\theta \mid y^{(k-1)}),
$$

the shape of the likelihood directly affects how much the posterior changes.

### Large discrimination parameter (\(a_k\))

When \(a_k\) is large,

- the logistic curve becomes steeper around \(\theta=b_k\),
- the response provides a large amount of information,
- the posterior distribution becomes much narrower,
- posterior variance decreases rapidly,
- confidence in the estimated ability increases.

### Small discrimination parameter (\(a_k\))

When \(a_k\) is small,

- the logistic curve changes gradually,
- the response provides less information,
- the posterior distribution changes only slightly,
- posterior variance decreases slowly,
- uncertainty remains relatively high.

Therefore, highly discriminative items reduce uncertainty much faster than weakly
discriminative items.

# Task 6 — Numerical Implementation of a Running Grid

Since the posterior distribution of the 2PL Item Response Theory model has no closed-form
analytical solution, it is approximated numerically using a fixed grid of latent ability
values.

## Algorithm

### Step 1 — Construct the Grid

Create a dense grid of latent ability values.

```python
theta = np.linspace(-5, 5, 500)
```

### Step 2 — Initialize the Prior

Evaluate the standard normal prior.

```python
prior = stats.norm.pdf(theta, 0, 1)
```

### Step 3 — Compute the Likelihood

For each item with discrimination parameter \(a_k\), difficulty parameter \(b_k\),
and observed response \(y_k\),

```python
p_grid = p_i(theta, a_k, b_k)
```

Compute the likelihood.

```python
likelihood = (p_grid**y_k) * ((1-p_grid)**(1-y_k))
```

### Step 4 — Update the Posterior

Multiply the current posterior by the likelihood.

```python
unnormalized_posterior = current_posterior * likelihood
```

### Step 5 — Normalize

Normalize numerically using the trapezoidal rule.

```python
Z = np.trapezoid(unnormalized_posterior, theta)

current_posterior = unnormalized_posterior / Z
```

### Step 6 — Compute Point Estimates

Posterior Mean

```python
theta_bayes = np.trapezoid(theta * current_posterior, theta)
```

MAP Estimate

```python
theta_map = theta[np.argmax(current_posterior)]
```

### Step 7 — Repeat

Use the normalized posterior as the prior for the next observation and repeat the
Bayesian updating procedure after each new response.

This grid-based method provides an accurate numerical approximation when no conjugate
closed-form posterior exists.

# Task 7 — Evaluating Convergence over the Timeline

The Bayesian estimation procedure is evaluated by simulating user responses to
20 items.

The simulation assumes

- True latent ability:

$$
\theta_{\mathrm{true}}=0.75
$$

- Number of items:

$$
n=20
$$

- Item discrimination parameters:

$$
a_k \sim \mathrm{Uniform}(0.5,2.0)
$$

- Item difficulty parameters:

$$
b_k \sim N(0,1)
$$

At each step,

1. Compute the probability of a correct response using the 2PL model.
2. Generate a simulated response from the Bernoulli distribution.
3. Update the posterior distribution using the grid-based Bayesian algorithm.
4. Compute the running Posterior Mean

$$
\hat{\theta}_{\mathrm{Bayes}}
$$

and the running MAP estimate

$$
\hat{\theta}_{\mathrm{MAP}}.
$$

Finally, Plotly is used to visualize the evolution of both estimators together with
a horizontal reference line representing the true latent ability.

## Analysis

During the early stages of the assessment, only a few responses are available.
Consequently, the posterior distribution remains relatively wide, and both the Posterior
Mean and MAP estimates may fluctuate because each new response has a strong influence on
the estimated ability.

As more responses are collected, additional evidence accumulates and the posterior
distribution becomes increasingly concentrated. The influence of the initial normal prior
gradually decreases, while the observed data dominate the estimation process.

Consequently,

- both the Posterior Mean and MAP estimate converge toward the true latent ability,
- the difference between the two estimators becomes progressively smaller,
- posterior uncertainty decreases,
- confidence in the estimated ability increases.

By the end of the simulation, both estimators approach the true ability value

$$
\theta_{\mathrm{true}}=0.75,
$$

demonstrating the consistency and effectiveness of sequential Bayesian estimation.

# Q2. Bayesian Tracking of Click-Through Rates (CTR) via Conjugate Beta-Binomial Updates

## Setup

An e-commerce platform continuously estimates the Click-Through Rate (CTR) of a newly launched advertisement.

Each user interaction is modeled as

$$
Y_k \in \{0,1\},
$$

where

- **\(Y_k = 1\)** : User clicks the advertisement.
- **\(Y_k = 0\)** : User does not click the advertisement.

The unknown click-through rate is denoted by

$$
\Theta=\theta,\qquad 0\le\theta\le1.
$$

Conditioned on \(\Theta=\theta\),

$$
Y_k \mid \Theta=\theta \sim \mathrm{Bernoulli}(\theta),
$$

so that

$$
P(Y_k=1\mid\Theta=\theta)=\theta.
$$

The prior distribution is

$$
\Theta\sim\mathrm{Beta}(\alpha_0,\beta_0).
$$

After every observation, the posterior distribution becomes the prior for the next Bayesian update.

# Task 1 — Structural Probability and Properties

The Beta probability density function is visualized for three different parameter settings:

- **Beta(1,1)** – Uniform (Uninformative Prior)
- **Beta(2,8)** – Right-skewed Distribution
- **Beta(8,2)** – Left-skewed Distribution

## Interpretation

The Beta distribution is controlled by two positive shape parameters,
\(\alpha\) and \(\beta\).

Its mean is

$$
E[\Theta]=\frac{\alpha}{\alpha+\beta}.
$$

### Beta(1,1)

When

$$
\alpha=\beta=1,
$$

the distribution is uniform over the interval \([0,1]\). This indicates that every value of the CTR is considered equally likely before observing any user interactions.

### Beta(2,8)

When

$$
\beta>\alpha,
$$

the distribution is concentrated near zero, representing the belief that the advertisement is unlikely to receive many clicks.

### Beta(8,2)

When

$$
\alpha>\beta,
$$

the distribution is concentrated near one, representing the belief that the advertisement is likely to receive many clicks.

In general,

- increasing \(\alpha\) shifts the distribution toward larger values of \(\theta\),
- increasing \(\beta\) shifts the distribution toward smaller values of \(\theta\),
- increasing both parameters while maintaining their ratio makes the distribution more concentrated, representing stronger prior confidence.

# Task 2 — Sequential Likelihood and Joint History

For a single user interaction, the likelihood function is

$$
L(y_k\mid\theta)
=
\theta^{y_k}
(1-\theta)^{1-y_k}.
$$

Assuming conditional independence of observations given \(\Theta=\theta\), the likelihood of the complete interaction history

$$
y^{(k)}
=
(y_1,y_2,\ldots,y_k)
$$

is

$$
L(y^{(k)}\mid\theta)
=
\prod_{i=1}^{k}
\theta^{y_i}
(1-\theta)^{1-y_i}.
$$

If

$$
C_k=\sum_{i=1}^{k}y_i
$$

denotes the total number of clicks, then the likelihood simplifies to

$$
L(y^{(k)}\mid\theta)
=
\theta^{C_k}
(1-\theta)^{k-C_k}.
$$

Thus, the likelihood depends only on the total numbers of clicks and non-clicks.

# Task 3 — Closed-Form Analytical Updates (Conjugacy)

Using Bayes' theorem,

$$
f(\theta\mid y^{(k)})
\propto
L(y_k\mid\theta)
\,f(\theta\mid y^{(k-1)}).
$$

Suppose the previous posterior distribution is

$$
\Theta\mid Y^{(k-1)}
\sim
\mathrm{Beta}(\alpha_{k-1},\beta_{k-1}).
$$

Its probability density function is

$$
f(\theta\mid y^{(k-1)})
\propto
\theta^{\alpha_{k-1}-1}
(1-\theta)^{\beta_{k-1}-1}.
$$

Multiplying the Beta prior by the Bernoulli likelihood gives

$$
\begin{aligned}
f(\theta\mid y^{(k)})
&\propto
\theta^{y_k}
(1-\theta)^{1-y_k}
\theta^{\alpha_{k-1}-1}
(1-\theta)^{\beta_{k-1}-1} \\
&=
\theta^{(\alpha_{k-1}+y_k)-1}
(1-\theta)^{(\beta_{k-1}+1-y_k)-1}.
\end{aligned}
$$

Therefore, the posterior distribution remains a Beta distribution,

$$
\Theta\mid Y^{(k)}
\sim
\mathrm{Beta}(\alpha_k,\beta_k),
$$

where

$$
\alpha_k=\alpha_{k-1}+y_k,
$$

$$
\beta_k=\beta_{k-1}+1-y_k.
$$

Hence, the Beta prior is conjugate to the Bernoulli likelihood, allowing exact Bayesian updates without numerical integration.

The posterior mean is

$$
E[\Theta\mid Y^{(k)}]
=
\frac{\alpha_k}{\alpha_k+\beta_k}.
$$

# Task 4 — Dynamic Shifting Mechanics

Each new observation updates the Beta distribution by modifying its shape parameters.

### Click Event (\(y_k=1\))

A click increases

$$
\alpha_k=\alpha_{k-1}+1,
$$

while \(\beta_k\) remains unchanged.

Consequently,

- the posterior distribution shifts toward higher values of \(\theta\),
- the posterior mean increases,
- the MAP estimate moves to the right,
- the estimated CTR becomes larger.

### No-Click Event (\(y_k=0\))

A non-click increases

$$
\beta_k=\beta_{k-1}+1,
$$

while \(\alpha_k\) remains unchanged.

Consequently,

- the posterior distribution shifts toward lower values of \(\theta\),
- the posterior mean decreases,
- the MAP estimate moves to the left,
- the estimated CTR becomes smaller.

As more observations are collected, the total concentration

$$
\alpha_k+\beta_k
$$

increases. Therefore, each additional observation has a progressively smaller influence on the posterior distribution.

Unlike the 2PL Item Response Theory model, the Beta-Binomial model is conjugate, allowing exact Bayesian updates without numerical approximation.

# Task 5 — Running Point Estimators

After each update,

$$
\Theta\mid Y^{(k)}
\sim
\mathrm{Beta}(\alpha_k,\beta_k).
$$

The Posterior Mean is

$$
\hat{\theta}_{\mathrm{Bayes}}^{(k)}
=
\frac{\alpha_k}{\alpha_k+\beta_k}.
$$

The Maximum A Posteriori (MAP) estimate is

$$
\hat{\theta}_{\mathrm{MAP}}^{(k)}
=
\frac{\alpha_k-1}{\alpha_k+\beta_k-2},
\qquad
\alpha_k>1,\;
\beta_k>1.
$$

If either parameter is less than or equal to one, the mode lies at the boundary of the interval \([0,1]\).

The Posterior Mean represents the expected CTR, whereas the MAP estimate represents the most probable CTR under the posterior distribution.

# Task 6 — Performance Tracking and Convergence Analysis

The Bayesian learning procedure is evaluated using a simulation of

- True CTR

$$
\theta_{\mathrm{true}}=0.35,
$$

- Number of impressions

$$
n=100,
$$

- Prior distribution

$$
\Theta\sim\mathrm{Beta}(1,1).
$$

For each impression,

1. Simulate a click or non-click using the Bernoulli distribution.
2. Update the Beta posterior using the conjugate formulas.
3. Compute the Posterior Mean and MAP estimate.
4. Visualize the posterior density and convergence using Plotly.

## Analysis

Initially, only a few observations are available, so each click or non-click produces noticeable changes in the posterior distribution.

As the number of impressions increases,

- the influence of the prior distribution gradually decreases,
- the posterior variance becomes smaller,
- the Posterior Mean and MAP estimate converge toward one another,
- both estimates approach the true CTR,

$$
\theta_{\mathrm{true}}=0.35.
$$

This demonstrates that Bayesian estimation becomes increasingly accurate as more data are collected.